In [1]:
!pip install xgboost

In [ ]:
import numpy as np
import pandas as pd
import random
from collections import defaultdict

# Importa a função preditiva oficial que você estruturou e o seu modelo já treinado
from main import simular_confronto

# Define variáveis globais do experimento (Aumentado para 10.000 para convergência ideal)
ANO_COPA_ATUAL = 2026
N_SIMULACOES = 10000

print(f"Módulos carregados com sucesso para a simulação da Copa de {ANO_COPA_ATUAL}!")


[SUCESSO] XGBoost treinado puramente com 7 características matemáticas.
Módulos carregados com sucesso para a simulação da Copa de 2026!


In [3]:
# Lista oficializada com as 48 seleções divididas nos 12 grupos (A até L)
# Nomes revisados para bater estritamente com os arquivos de dados do modelo
grupos_2026 = {
    'Grupo A': ['Mexico', 'South Africa', 'South Korea', 'Czechia'],
    'Grupo B': ['Canada', 'Bosnia and Herzegovina', 'Qatar', 'Switzerland'],
    'Grupo C': ['Brazil', 'Morocco', 'Haiti', 'Scotland'],
    'Grupo D': ['United States', 'Paraguay', 'Australia', 'Turkey'],
    'Grupo E': ['Germany', 'Curacao', 'Ivory Coast', 'Ecuador'],
    'Grupo F': ['Netherlands', 'Japan', 'Sweden', 'Tunisia'],
    'Grupo G': ['Belgium', 'Egypt', 'Iran', 'New Zealand'],
    'Grupo H': ['Spain', 'Cape verde', 'Saudi Arabia', 'Uruguay'],
    'Grupo I': ['France', 'Senegal', 'Iraq', 'Norway'],
    'Grupo J': ['Argentina', 'Algeria', 'Austria', 'Jordan'],
    'Grupo K': ['Portugal', 'DR Congo', 'Uzbekistan', 'Colombia'],
    'Grupo L': ['England', 'Croatia', 'Ghana', 'Panama']
}

print(f"Tabela de grupos configurada. Total de seleções: {sum(len(v) for v in grupos_2026.values())}")

Tabela de grupos configurada. Total de seleções: 48


In [4]:
def sortear_resultado(prob_casa, prob_empate, prob_fora, permitir_empate=True):
    """
    Executa a amostragem de Monte Carlo baseada nas probabilidades do modelo.
    0 = Vitória do Time da Casa
    1 = Empate
    2 = Vitória do Time de Fora
    """
    # Garante que a soma das probabilidades seja estritamente 1.0 (evita problemas de float do Pandas)
    soma_total = prob_casa + prob_empate + prob_fora
    if soma_total > 0:
        prob_casa = prob_casa / soma_total
        prob_empate = prob_empate / soma_total
        prob_fora = prob_fora / soma_total

    if permitir_empate:
        return np.random.choice([0, 1, 2], p=[prob_casa, prob_empate, prob_fora])
    else:
        # Lógica para o Mata-Mata: Elimina o empate redistribuindo a proporção uniformemente
        soma = prob_casa + prob_fora
        if soma == 0:
            return np.random.choice([0, 2]) # Fallback de segurança se houver erro crítico de dados
            
        prob_casa_mm = prob_casa / soma
        prob_fora_mm = prob_fora / soma
        return np.random.choice([0, 2], p=[prob_casa_mm, prob_fora_mm])

In [ ]:
def rodar_uma_copa(ano_copa):
    fases_alcancadas = {}
    
    primeiros_colocados = []
    segundos_colocados = []
    pontuacao_grupos = defaultdict(lambda: defaultdict(lambda: {'pontos': 0}))
    
    # FASE DE GRUPOS: simula todos os confrontos dentro de cada grupo
    for nome_grupo, selecoes in grupos_2026.items():
        pts = {time: 0 for time in selecoes}
        
        for i in range(len(selecoes)):
            for j in range(i + 1, len(selecoes)):
                tc = selecoes[i]
                tf = selecoes[j]
                
                res_funcao = simular_confronto(ano_copa, tc, tf)
                p_casa, p_empate, p_fora = res_funcao if isinstance(res_funcao, tuple) else (0.33, 0.34, 0.33)
                
                resultado = sortear_resultado(p_casa, p_empate, p_fora, permitir_empate=True)
                
                if resultado == 0:
                    pts[tc] += 3
                elif resultado == 1:
                    pts[tc] += 1
                    pts[tf] += 1
                elif resultado == 2:
                    pts[tf] += 3
                    
        ranking_grupo = sorted(pts.items(), key=lambda x: x[1], reverse=True)
        
        primeiros_colocados.append(ranking_grupo[0][0])
        segundos_colocados.append(ranking_grupo[1][0])
        pontuacao_grupos[nome_grupo]['terceiro'] = ranking_grupo[2]

    # Repescagem: Seleciona os 8 melhores 3º colocados no geral
    terceiros = [dados['terceiro'] for dados in pontuacao_grupos.values()]
    terceiros_ordenados = sorted(terceiros, key=lambda x: x[1], reverse=True)
    melhores_terceiros = [time for time, pontos in terceiros_ordenados[:8]]
    
    # Chave de mata-mata seguro (garante exatamente 32 times em pares)
    random.shuffle(primeiros_colocados)
    random.shuffle(segundos_colocados)
    random.shuffle(melhores_terceiros)
    
    classificados_fase_32 = []
    
    # Casamento Perfeito: 12 Primeiros contra 12 Segundos
    for idx in range(12):
        classificados_fase_32.append(primeiros_colocados[idx])
        classificados_fase_32.append(segundos_colocados[idx])
        
    # Os 8 melhores Terceiros entram contra os 8 Segundos que restaram do sorteio
    # (Usamos fatiamento seguro para garantir que a lista tenha exatamente 32 elementos)
    segundos_restantes = segundos_colocados[12:] if len(segundos_colocados) > 12 else segundos_colocados[:8]
    for idx in range(min(8, len(melhores_terceiros))):
        classificados_fase_32.append(melhores_terceiros[idx])
        classificados_fase_32.append(segundos_colocados[(idx + 4) % len(segundos_colocados)])
    
    # Garante que a lista final tenha tamanho par caso falte algum dado no ciclo
    if len(classificados_fase_32) % 2 != 0:
        classificados_fase_32.pop()

    # Define quem pisou na primeira fase eliminatória
    for time in classificados_fase_32:
        fases_alcancadas[time] = 'Dezesseis-avos'

    # mata-mata recursivo
    def simular_fase_mata_mata(participantes, nome_fase):
        vencedores = []
        for i in range(0, len(participantes) - 1, 2):  # Passo de segurança (len - 1)
            tc = participantes[i]
            tf = participantes[i+1]
            
            res_funcao = simular_confronto(ano_copa, tc, tf)
            p_casa, p_empate, p_fora = res_funcao if isinstance(res_funcao, tuple) else (0.5, 0.0, 0.5)
            
            resultado = sortear_resultado(p_casa, p_empate, p_fora, permitir_empate=False)
            vencedor = tc if resultado == 0 else tf
            
            vencedores.append(vencedor)
            fases_alcancadas[vencedor] = nome_fase
        return vencedores

    # Execução das chaves eliminatórias sucessivas
    oitavas = simular_fase_mata_mata(classificados_fase_32, 'Oitavas') 
    quartas = simular_fase_mata_mata(oitavas, 'Quartas')               
    semis = simular_fase_mata_mata(quartas, 'Semi')                    
    finalistas = simular_fase_mata_mata(semis, 'Final')                
    
    # A Grande Final (Garante que chegaram 2 times)
    if len(finalistas) >= 2:
        tc, tf = finalistas[0], finalistas[1]
        res_final = simular_confronto(ano_copa, tc, tf)
        p_casa, p_empate, p_fora = res_final if isinstance(res_final, tuple) else (0.5, 0.0, 0.5)
        
        vencedor_final = sortear_resultado(p_casa, p_empate, p_fora, permitir_empate=False)
        campeao = tc if vencedor_final == 0 else tf
        fases_alcancadas[campeao] = 'Campeao'
    
    return fases_alcancadas

In [6]:
# Dicionário acumulador estruturado
historico_estatistico = defaultdict(lambda: {
    'Dezesseis-avos': 0, 'Oitavas': 0, 'Quartas': 0, 'Semi': 0, 'Final': 0, 'Campeao': 0
})

print(f"Iniciando {N_SIMULACOES} simulações completas da Copa...")

for rodada in range(1, N_SIMULACOES + 1):
    if rodada % 2000 == 0:
        print(f"-> {rodada} torneios processados...")
        
    resultado_torneio = rodar_uma_copa(ano_copa=ANO_COPA_ATUAL)
    
    # Incrementa os dados do histórico de forma isolada inicialmente
    for pais, fase in resultado_torneio.items():
        historico_estatistico[pais][fase] += 1

print("Simulações finalizadas! Processando os dados obtidos...")

Iniciando 1000 simulações completas da Copa...
Simulações finalizadas! Processando os dados obtidos...


In [ ]:
# Converte o dicionário em DataFrame e preenche seleções ausentes com zero
df_finais = pd.DataFrame.from_dict(historico_estatistico, orient='index').fillna(0)

# Correção matemática: Soma cumulativa reversa para contabiilizar mérito das fases anteriores
df_finais['Final'] = df_finais['Final'] + df_finais['Campeao']
df_finais['Semi'] = df_finais['Semi'] + df_finais['Final']
df_finais['Quartas'] = df_finais['Quartas'] + df_finais['Semi']
df_finais['Oitavas'] = df_finais['Oitavas'] + df_finais['Quartas']
df_finais['Dezesseis-avos'] = df_finais['Dezesseis-avos'] + df_finais['Oitavas']

# Transforma as frequências brutas em probabilidades percentuais (%)
df_probabilidades = (df_finais / N_SIMULACOES) * 100

# Ordenação lógica das colunas por avanço de fase
colunas_cronologicas = ['Dezesseis-avos', 'Oitavas', 'Quartas', 'Semi', 'Final', 'Campeao']
df_probabilidades = df_probabilidades[colunas_cronologicas]

# Ordena as equipes de cima para baixo por quem obteve mais títulos de campeão de verdade
df_probabilidades = df_probabilidades.sort_values(by='Campeao', ascending=False)

# Exibe o resultado final formatado de forma limpa e impecável
print(f"\n================ PROBABILIDADES DA COPA DO MUNDO {ANO_COPA_ATUAL} ================")
display(df_probabilidades.round(2))


================ PROBABILIDADES DA COPA DO MUNDO 2026 ================


,Dezesseis-avos,Oitavas,Quartas,Semi,Final,Campeao
Argentina,86.1,64.9,47.0,33.8,20.0,14.6
Japan,87.7,57.6,39.1,24.2,12.0,6.9
France,77.0,53.6,38.1,25.4,11.6,6.3
Germany,92.3,59.9,36.7,22.6,9.9,5.8
United States,82.3,50.9,30.7,17.1,8.4,5.5
Portugal,84.9,55.1,34.0,19.8,8.2,5.1
Croatia,79.5,56.7,38.8,23.2,9.5,4.7
Turkey,81.5,50.8,31.8,18.5,7.5,4.1
Brazil,82.6,55.4,33.9,18.7,8.5,4.1
Morocco,96.2,55.7,34.5,19.9,8.0,4.0
